# Geospatial Data Analysis and Visualization
## Class 08 — Foundations of Spatial Data & Environments

**Keynote Speaker:** Md Wabidur Rahman, Researcher, Research and Integrated Thought (RIT)
**Class 08 · 16 October 2026**

---

### What this notebook covers

| Part | Topic | Main tool |
|------|------|---------|
| 0 | **Preparing the environment** — Anaconda, JupyterLab, libraries | `conda`, `pip` |
| 1 | **What spatial data is** — vector versus raster | concepts |
| 2 | **Vector formats** — Shapefile, GeoJSON, GeoParquet, KML | `geopandas` |
| 3 | **Geometry objects** — Point, Line, Polygon | `shapely` |
| 4 | **Raster formats** — NetCDF, GeoTIFF | `xarray`, `rioxarray` |
| 5 | **A first map** — vector and raster together | `matplotlib` |

---

### Today's research question

> **"How have air temperature and relative humidity changed over Bangladesh between 2000 and 2024,
> and how does that change vary from place to place?"**

By the end of the four notebooks you will be able to answer that question with analysis and
figures of **journal-submission quality**. This notebook lays the **foundation** —
knowing the data, reading it, and drawing a first map.

---

### Our data (four datasets)

| # | Dataset | What it is | Source |
|---|------|-----|------|
| 1 | `monthly_mean_data` | ERA5 monthly means — temperature and humidity, 300 months | ECMWF (Copernicus) |
| 2 | `hourly_mean_data` | ERA5 hourly means — the diurnal cycle | ECMWF (Copernicus) |
| 3 | `BGD_adm` | Administrative boundaries of Bangladesh (GADM, 2015) | gadm.org |
| 4 | `bgd_adm_bbs_20201113` | Official boundaries of Bangladesh (BBS/HDX, 2020) | data.humdata.org |

---
# Part 0 — Environment Setup

The geospatial libraries use **GDAL, GEOS and PROJ**, written in C/C++, underneath.
Those three are the engine of the whole GIS world —

- **GDAL** — reading and writing files in any format
- **GEOS** — geometric computation (intersection, buffer, area)
- **PROJ** — converting from one coordinate system to another

If their versions do not match, the installation breaks. That is why building a separate
environment with **Anaconda** is the safest route.

### How to build the environment on your own machine

Run these commands in a terminal / Anaconda Prompt (once only) —

```bash
# 1) A new environment (from conda-forge, because GDAL/GEOS/PROJ are best there)
conda create -n geo -c conda-forge python=3.11 ^
      geopandas rasterio rioxarray xarray netcdf4 shapely pyproj ^
      jupyterlab matplotlib folium mapclassify contextily ^
      scikit-learn esda libpysal splot pyarrow -y

# 2) Activate it
conda activate geo

# 3) Open JupyterLab
jupyter lab
```

> **Why `conda-forge`?** `pip install geopandas` very often breaks GDAL on Windows.
> conda-forge ships all the binaries matched to one another — which makes it the right choice.

> **A quick alternative:** without `conda`, `pip install geopandas rasterio rioxarray
> xarray netCDF4 folium mapclassify esda libpysal scikit-learn pyarrow` works in most cases
> these days (this notebook itself runs under pip).

In [ ]:
# -- Libraries and shared styling (identical in every notebook) ------------
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import geopandas as gpd, xarray as xr

# Publication-ready defaults so the figures can go straight into a journal
plt.rcParams.update({
    "figure.dpi": 110,          # for viewing on screen
    "savefig.dpi": 300,         # for saving to a file (journals usually want >=300)
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.autolayout": False,
})

# Brand colours — the same in every notebook
C_WARM, C_COOL, C_ACC, C_INK = "#B4562F", "#3D7EA6", "#E0A458", "#1B3A5C"

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("geopandas   ", gpd.__version__)
print("xarray      ", xr.__version__)
print("matplotlib  ", matplotlib.__version__)

### What each library does

| Library | Role | In one line |
|-----------|-----|----------|
| `geopandas` | vector data (a table plus geometry) | **pandas + maps** |
| `shapely` | individual geometry objects and their arithmetic | the mathematics of points, lines and polygons |
| `pyproj` | coordinate system conversion | the CRS translator |
| `xarray` | multi-dimensional grids (time x latitude x longitude) | **the N-dimensional form of pandas** |
| `rasterio` | reading and writing raster files | the GeoTIFF workhorse |
| `rioxarray` | gives xarray its geographic powers | xarray + CRS + clip |
| `matplotlib` | static figures | publication figures |
| `folium` | interactive web maps | maps in the browser |

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  SETUP — locating and preparing the data  (identical in every notebook)
# ════════════════════════════════════════════════════════════════════
# This cell finds your four datasets by itself:
#   * monthly_mean_data_*.zip / data_0.nc   (ERA5 monthly)
#   * hourly_mean_data_*.zip  / data_0.nc   (ERA5 hourly)
#   * BGD_adm.zip                           (GADM shapefile)
#   * bgd_adm_bbs_*_shp.zip                 (BBS/HDX shapefile)
# If it finds the zips it extracts them into ./data/. If they are already extracted it uses them as they are.

from pathlib import Path
import zipfile, sys, os

DATA = Path.cwd() / "data"
OUT  = Path.cwd() / "outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

# Where to look
SEARCH_ROOTS = [Path.cwd(), Path.cwd()/"data", Path.cwd().parent,
                Path.home()/"Downloads", Path.home()/"Desktop", Path.home()]

def _find(patterns, roots=SEARCH_ROOTS, maxdepth=3):
    """Return the first file that matches the pattern."""
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            try:
                hits = sorted(root.glob(pat)) + sorted(root.glob("*/" + pat))
                if maxdepth >= 3:
                    hits += sorted(root.glob("*/*/" + pat))
                for h in hits:
                    if h.exists():
                        return h
            except Exception:
                pass
    return None

def _ensure(zip_patterns, member_pattern, dest):
    """If member_pattern is not in dest, find the zip and extract it there."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    got = sorted(dest.rglob(member_pattern))
    if got:
        return got[0]
    z = _find(zip_patterns)
    if z is not None:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        got = sorted(dest.rglob(member_pattern))
        if got:
            print(f"  extracted {z.name}  ->  {dest.name}/")
            return got[0]
    return None

print("Looking for the data ...")
NC_MONTHLY = _ensure(["*monthly_mean*zip"], "*.nc", DATA/"nc_monthly")
NC_HOURLY  = _ensure(["*hourly_mean*zip"],  "*.nc", DATA/"nc_hourly")
SHP_GADM   = _ensure(["*BGD_adm*.zip"], "BGD_adm2.shp", DATA/"shp_adm")
SHP_BBS    = _ensure(["*bgd_adm_bbs*shp*.zip", "*bgd_adm_bbs*.zip"],
                     "bgd_admbnda_adm2*.shp", DATA/"shp_bbs")

# GADM is optional -- only the GADM versus BBS comparison in Class 08 needs
# it (see the README).
status = {"ERA5 monthly (.nc)": NC_MONTHLY, "ERA5 hourly (.nc)": NC_HOURLY,
          "BBS/HDX shapefile": SHP_BBS}
print()
ok = True
for k, v in status.items():
    print(f"  {'✓' if v else '✗'}  {k:24s} {v if v else 'not found'}")
    ok = ok and (v is not None)
print(f"  {'✓' if SHP_GADM else '·'}  {'GADM (optional)':24s} "
      f"{SHP_GADM if SHP_GADM else 'absent -- only the Class 08 comparison is skipped'}")
print()
if ok:
    print("All required data ready")
else:
    print("Some required files were not found — put the zip files next to this notebook and run again.")

> **What if a file is not found?** The cell above looks for your four zip files in the
> notebook's folder, in `data/`, and in `Downloads`. If it does not find them,
> put the four zips **next to this notebook** and run the cell again — it will
> extract them into `data/` by itself.

---
# Part 1 — What is spatial data? (Vector versus Raster)

All geographic data falls into two families —

| | **Vector** | **Raster** |
|---|---|---|
| Structure | points, lines, polygons | a grid of equally sized cells (pixels) |
| Examples | district boundaries, rivers, school locations | satellite images, a temperature grid |
| Files | `.shp`, `.geojson`, `.gpkg`, `.parquet` | `.tif`, `.nc`, `.img` |
| What it does well | precise boundaries, attribute tables | continuous variation (heat, elevation) |
| Python tools | `geopandas`, `shapely` | `xarray`, `rasterio`, `rioxarray` |

**Our project has both** — the district boundaries of Bangladesh (vector) and the
ERA5 temperature grid (raster). Real research happens **where the two meet**:
"what is the mean temperature in Rangamati district?" needs both the vector (the district)
and the raster (the temperature) together.

In [ ]:
# the difference between the two families at a glance — a small figure
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))

# left: vector — a few points, a line, a polygon
ax[0].add_patch(plt.Polygon([(1,1),(4,1),(4,3.2),(2.2,3.8),(1,2.6)],
                            fc="#cfe3f7", ec=C_INK, lw=1.6))
ax[0].plot([0.5,2,3.5,4.6],[4.4,4.0,4.6,4.1], "-", color=C_WARM, lw=2)
ax[0].plot([1.6,3.0,3.8],[2.0,2.4,1.6], "o", color=C_WARM, ms=8)
ax[0].set_title("Vector — points, line, polygon"); ax[0].set_xlim(0,5); ax[0].set_ylim(0,5)

# right: raster — a grid of numbers
rng = np.random.RandomState(7)
grid = rng.rand(6, 6) * 10 + 20
im = ax[1].imshow(grid, cmap="RdYlBu_r")
for i in range(6):
    for j in range(6):
        ax[1].text(j, i, f"{grid[i,j]:.0f}", ha="center", va="center", fontsize=7)
ax[1].set_title("Raster — a grid of pixel values")
plt.colorbar(im, ax=ax[1], shrink=0.8, label="value")
for a in ax: a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()

---
# Part 2 — Reading and writing vector formats

## 2.1 Reading a Shapefile — our two boundary datasets

A **Shapefile** is not one file but **a set of files** —

| Extension | What it holds | Required? |
|-----------|---------|---------------|
| `.shp` | the geometry (the shapes) | yes |
| `.shx` | the geometry index | yes |
| `.dbf` | the attribute table (names, codes) | yes |
| `.prj` | the coordinate system (CRS) | without it the CRS is unknown |
| `.cpg` | the character encoding | optional |

> **⚠ The most common mistake:** copying only the `.shp` file. Without the others
> it is **useless**. This is why GeoPackage or GeoParquet is better for modern work — one file holds everything.

In [ ]:
# GADM (2015) — reading the four levels
gadm = {}
if SHP_GADM is None:
    # GADM's licence forbids redistribution, so the file is not in this
    # repository.  Download the Bangladesh shapefile from gadm.org and put it
    # next to this notebook as BGD_adm.zip (see the README).  Everything else
    # in the course runs without it.
    print("GADM not found — the GADM versus BBS comparison will be skipped.")
    print("See 'Getting the GADM file' in the README.")
else:
    for lv in range(4):
        p = SHP_GADM.parent / f"BGD_adm{lv}.shp"
        gadm[lv] = gpd.read_file(p)

    print("GADM (gadm.org, 2015)")
    for lv, g in gadm.items():
        print(f"  adm{lv}:  {len(g):5d} feature  |  CRS = {g.crs.to_string()}")

gadm[2].head(3)[["NAME_1", "NAME_2", "TYPE_2", "geometry"]] if gadm else None

In [ ]:
# BBS / HDX (2020) — the official boundaries of the Government of Bangladesh
BBSDIR = SHP_BBS.parent
bbs = {}
for lv in range(5):
    hits = sorted(BBSDIR.glob(f"bgd_admbnda_adm{lv}_*.shp"))
    if hits:
        bbs[lv] = gpd.read_file(hits[0])

print("BBS / HDX (data.humdata.org, 2020)")
names = {0: "Country", 1: "Division", 2: "District",
         3: "Upazila", 4: "Union"}
for lv, g in bbs.items():
    print(f"  adm{lv}:  {len(g):5d} feature   {names[lv]}")

bbs[2].head(3)[["ADM1_EN", "ADM2_EN", "ADM2_PCODE", "geometry"]]

### ⚠ Two sources, two different counts — an important lesson for research

Compare the results above —

| Level | GADM (2015) | BBS (2020) |
|------|-------------|-------------|
| Divisions | **7** | **8** |
| Districts | **65** | **64** |
| Upazilas | 463 | 544 |

**Why do they differ?**
- **Mymensingh** was created as the eighth division in 2015 — GADM does not have it.
- There are some inconsistencies in GADM's district count; BBS is the **official, current** source.

> **The rule for a journal:** which boundaries you used must be stated clearly in
> **Methods** — the source, the version, the date. For example:
> *"Administrative boundaries were obtained from the Bangladesh Bureau of
> Statistics (BBS) via HDX, version 2020-11-13 (64 districts, 8 divisions)."*
>
> **In this course we use BBS (2020)** — because it is official and current.

In [ ]:
# side by side — the same country, two sources
if not gadm:
    print("GADM is absent — showing the BBS boundaries only.")
    fig, ax0 = plt.subplots(figsize=(6, 5))
    bbs[1].plot(ax=ax0, edgecolor="white", linewidth=0.8, color=C_WARM, alpha=0.85)
    ax0.set_title(f"BBS 2020 — {len(bbs[1])} divisions")
    ax0.set_xlabel("Longitude (°E)"); ax0.set_ylabel("Latitude (°N)")
    plt.tight_layout(); plt.show()
else:
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))

    gadm[1].plot(ax=ax[0], edgecolor="white", linewidth=0.8, color=C_COOL, alpha=0.85)
    ax[0].set_title(f"GADM 2015 — {len(gadm[1])} divisions")

    bbs[1].plot(ax=ax[1], edgecolor="white", linewidth=0.8, color=C_WARM, alpha=0.85)
    ax[1].set_title(f"BBS 2020 — {len(bbs[1])} divisions")

    for a in ax:
        a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
        a.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

print("BBS divisions:", ", ".join(sorted(bbs[1]["ADM1_EN"])))

## 2.2 Writing to other formats — and which to use when

Now we write the same district data to several formats and measure the **size and speed**.
This choice matters in a real project — the wrong format means slow work and large files.

In [ ]:
import time

dist = bbs[2][["ADM2_EN", "ADM1_EN", "ADM2_PCODE", "geometry"]].copy()
rows = []

targets = [
    ("Shapefile",  OUT/"districts.shp",     dict()),
    ("GeoJSON",    OUT/"districts.geojson", dict()),
    ("GeoPackage", OUT/"districts.gpkg",    dict(layer="districts")),
    ("FlatGeobuf", OUT/"districts.fgb",     dict()),
    ("GeoParquet", OUT/"districts.parquet", dict()),
]

for name, path, kw in targets:
    t0 = time.time()
    if name == "GeoParquet":
        dist.to_parquet(path)
    else:
        dist.to_file(path, **kw)
    t_write = time.time() - t0

    t0 = time.time()
    back = gpd.read_parquet(path) if name == "GeoParquet" else gpd.read_file(path)
    t_read = time.time() - t0

    rows.append({"format": name,
                 "size_MB": round(path.stat().st_size/1e6, 2),
                 "write_s": round(t_write, 3),
                 "read_s": round(t_read, 3),
                 "rows_back": len(back)})

pd.DataFrame(rows).set_index("format")

### How to read the results

- **GeoJSON** is the largest — because it is **plain text**, with every number written out
  as characters. Good for the web, bad for large data.
- **GeoParquet** is the smallest and the fastest to read — modern, compressed, column-oriented.
- **Shapefile / FlatGeobuf / GeoPackage** sit in between, but GeoPackage and
  FlatGeobuf keep everything in **a single file** (not the four or five of a Shapefile).

| When to use which |  |
|---|---|
| Sending over the web / an API | **GeoJSON** |
| Your own analysis, large data | **GeoParquet** |
| Handing to a colleague, opening in QGIS | **GeoPackage** |
| Giving to older software | **Shapefile** (if you must) |
| Showing in Google Earth | **KML** |

> **Three limitations of the Shapefile:** (1) column names are capped at **10 characters**,
> (2) one geometry type per file, (3) a maximum size of **2 GB**.

In [ ]:
# KML — for opening in Google Earth
kml_path = OUT/"districts.kml"
try:
    dist.to_file(kml_path, driver="KML")
    print(f"KML written  {kml_path.name}  ({kml_path.stat().st_size/1e6:.2f} MB)")
    print("-> it can be opened directly in Google Earth.")
except Exception as e:
    print("Could not write the KML:", str(e)[:120])
    print("Fix: GDAL's KML driver is needed (it ships with conda-forge).")

# the 10-character limit on column names — the famous Shapefile problem
demo = dist.head(3).copy()
demo["a_very_long_column_name"] = 1
demo.to_file(OUT/"trunc_demo.shp")
print("\nColumn names after writing to a Shapefile:")
print(list(gpd.read_file(OUT/"trunc_demo.shp").columns))
print("<- notice the name was truncated to ten characters!")

---
# Part 3 — Geometry Objects (`shapely`)

Every cell of the `geometry` column of a GeoDataFrame holds a **shapely** object.
These are the building blocks of vector data.

| Object | What it is | Example |
|-------|-----|---------|
| `Point` | a single location | a weather station |
| `LineString` | a connected sequence of points | a river, a road |
| `Polygon` | a closed area (it may have holes) | a district |
| `MultiPolygon` | several separate polygons together | a coastal district with islands |

In [ ]:
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely import wkt

# the approximate location of Dhaka
dhaka = Point(90.4125, 23.8103)

# an imaginary river line
river = LineString([(88.5, 24.8), (89.5, 24.0), (90.4, 23.5), (91.0, 22.8)])

# a rectangular area
box = Polygon([(89.0, 23.0), (91.5, 23.0), (91.5, 25.0), (89.0, 25.0)])

for name, g in [("Point", dhaka), ("LineString", river), ("Polygon", box)]:
    print(f"{name:12s} type={g.geom_type:10s} empty={g.is_empty}  valid={g.is_valid}")

print("\nWKT (Well-Known Text) — the written form of a geometry:")
print(" ", dhaka.wkt)
print(" ", box.wkt)

# from WKT back to an object
print("\nWKT → object:", wkt.loads("POINT (90.4125 23.8103)").geom_type)

In [ ]:
# geometric measures and relationships
print("=== Measures (in degrees — be careful already!) ===")
print(f"length of the line : {river.length:.4f}  (degrees)")
print(f"area of the polygon : {box.area:.4f}  (square degrees)")
print(f"perimeter : {box.length:.4f}")
print(f"centroid : {box.centroid.wkt}")
print(f"bounding box (minx, miny, maxx, maxy) : {box.bounds}")

print("\n=== Spatial relationships (predicates) — True/False ===")
print(f"does box contain Dhaka?    box.contains(dhaka)   = {box.contains(dhaka)}")
print(f"is Dhaka inside box?       dhaka.within(box)     = {dhaka.within(box)}")
print(f"does the river cross box?  river.intersects(box) = {river.intersects(box)}")
print(f"is the river fully inside? river.within(box)     = {river.within(box)}")

> **⚠ Extremely important:** the area above came out in **square degrees** —
> which has no physical meaning! One degree of longitude is about 111 km at the equator
> but almost 0 near the poles. To get a correct area in square kilometres you must convert to a
> **projected CRS** — the whole subject of **Notebook 2**.

In [ ]:
# geometric operations — building new shapes
buf   = dhaka.buffer(0.45)          # a circle around Dhaka
inter = box.intersection(buf)       # the part common to both
diff  = box.difference(buf)         # the circle subtracted from box
uni   = box.union(buf)              # the two combined

print(f"buffer area      : {buf.area:.4f}")
print(f"intersection area: {inter.area:.4f}")
print(f"difference area  : {diff.area:.4f}")
print(f"union area       : {uni.area:.4f}")
print(f"check: inter + diff = {inter.area + diff.area:.4f}  (= box area {box.area:.4f}) ✓")

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for a, geom, ttl, col in [
        (ax[0], buf,   "buffer(0.45)",   C_ACC),
        (ax[1], inter, "intersection",   C_WARM),
        (ax[2], diff,  "difference",     C_COOL),
        (ax[3], uni,   "union",          "#7FB069")]:
    gpd.GeoSeries([box]).plot(ax=a, fc="none", ec="gray", ls="--", lw=1)
    gpd.GeoSeries([geom]).plot(ax=a, fc=col, ec="k", alpha=0.75, lw=0.8)
    a.plot(*dhaka.xy, "k*", ms=9)
    a.set_title(ttl); a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
# MultiPolygon — why real data needs it
coastal = bbs[2][bbs[2]["ADM2_EN"].isin(["Bhola", "Barguna", "Noakhali", "Dhaka"])]
print("Geometry type (coastal districts contain islands):")
for _, r in coastal.iterrows():
    g = r.geometry
    parts = len(g.geoms) if g.geom_type == "MultiPolygon" else 1
    print(f"  {r['ADM2_EN']:12s} {g.geom_type:14s} parts = {parts:3d}")

fig, ax = plt.subplots(figsize=(6, 6))
coastal.plot(ax=ax, column="ADM2_EN", legend=True, edgecolor="k", linewidth=0.4,
             cmap="Set2", legend_kwds={"loc": "upper right", "fontsize": 8})
ax.set_title("MultiPolygon — islands are separate parts")
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

---
# Part 4 — Raster format: NetCDF

## 4.1 What is NetCDF?

**NetCDF** (Network Common Data Form) is the standard format for **multi-dimensional scientific
data** — climate, ocean and atmospheric data almost always arrive in it.

A GeoTIFF holds one image (rows x columns x bands).
A NetCDF can hold **time x height x latitude x longitude** — and several variables at once,
each with its own units and description.

Our file is **ERA5 reanalysis** — produced by the European weather agency ECMWF,
the most widely used climate dataset in the world.

> **What is a reanalysis?** Every past observation (stations, balloons, satellites, ships)
> is fed into a weather model to produce **gap-free, evenly gridded**
> data for the whole planet. So you get a value even where there is no station.

In [ ]:
# opening the NetCDF — with xarray
dsm = xr.open_dataset(NC_MONTHLY)
print(dsm)

### How to read the structure above

```
Dimensions:  (valid_time: 300, pressure_level: 1, latitude: 29, longitude: 29)
```
-> **300 time steps**, **1** pressure level, a **29 x 29** grid.

```
Coordinates:            <- the actual values of the axes
  * valid_time  2000-01-01 ... 2024-12-01
  * latitude    27.0 ... 20.0
  * longitude   87.0 ... 94.0
Data variables:         <- the actual measurements
    t  (valid_time, pressure_level, latitude, longitude)   <- temperature
    r  (valid_time, pressure_level, latitude, longitude)   <- relative humidity
```

- **`t`** = Temperature, in **kelvin (K)**
- **`r`** = Relative humidity, in **per cent (%)**
- **`pressure_level` = 1000 hPa** -> the layer just above the surface

In [ ]:
# the full description of each variable (metadata = the data's identity card)
for v in ["t", "r"]:
    a = dsm[v].attrs
    print(f"{v}:  {a.get('long_name')}  [{a.get('units')}]   "
          f"standard_name = {a.get('standard_name')}")

print("\nOverall information about the dataset:")
for k in ["institution", "Conventions", "GRIB_centreDescription"]:
    print(f"  {k:24s}: {dsm.attrs.get(k)}")

print("\n=== Extent of the grid ===")
print(f"  latitude  : {float(dsm.latitude.min()):.2f}N  ->  {float(dsm.latitude.max()):.2f}N")
print(f"  longitude : {float(dsm.longitude.min()):.2f}E  ->  {float(dsm.longitude.max()):.2f}E")
print(f"  resolution: {abs(float(dsm.latitude[1]-dsm.latitude[0])):.2f} deg  ~ "
      f"{abs(float(dsm.latitude[1]-dsm.latitude[0]))*111:.0f} km")
print(f"  time      : {pd.Timestamp(dsm.valid_time.values[0]):%Y-%m}  ->  "
      f"{pd.Timestamp(dsm.valid_time.values[-1]):%Y-%m}   ({dsm.sizes['valid_time']} months)")

In [ ]:
# squeeze — dropping the length-1 axis (pressure_level), and kelvin -> Celsius
ds = dsm.squeeze(drop=True)
ds["t_c"] = ds["t"] - 273.15
ds["t_c"].attrs = {"long_name": "Air temperature", "units": "°C"}

print("Shape (after squeeze):", dict(ds.sizes))
print()
print(f"temperature : {float(ds.t_c.min()):.2f} C  ->  {float(ds.t_c.max()):.2f} C   "
      f"(mean {float(ds.t_c.mean()):.2f} C)")
print(f"humidity    : {float(ds.r.min()):.2f} %  ->  {float(ds.r.max()):.2f} %    "
      f"(mean {float(ds.r.mean()):.2f} %)")
print(f"\nMissing values (NaN): t = {int(ds.t_c.isnull().sum())}, r = {int(ds.r.isnull().sum())}")
print("-> ERA5 reanalysis is completely gap-free, which is its great advantage.")

> **Why does the unit conversion matter?** ERA5 gives temperature in **kelvin**. A journal
> almost always wants **°C**. The formula: `°C = K − 273.15`. Forget this step and
> every result will be 273 degrees too high — a very common but serious mistake.

In [ ]:
# selecting data in xarray — three ways
print("1) .sel() — by actual value (a date, a coordinate)")
jan2020 = ds.t_c.sel(valid_time="2020-01-01")
print("   shape of the January 2020 map:", jan2020.shape)

print("\n2) .sel(method='nearest') — the nearest grid cell")
dhaka_t = ds.t_c.sel(latitude=23.81, longitude=90.41, method="nearest")
print(f"   nearest cell to Dhaka: lat={float(dhaka_t.latitude):.2f}, "
      f"lon={float(dhaka_t.longitude):.2f}")
print(f"   25-year mean temperature there: {float(dhaka_t.mean()):.2f} C")

print("\n3) .isel() — by position (index)")
print("   first time step:", pd.Timestamp(ds.isel(valid_time=0).valid_time.values).date())

print("\n4) slice — a range of time")
recent = ds.t_c.sel(valid_time=slice("2020-01-01", "2024-12-01"))
print(f"   2020-2024: {recent.sizes['valid_time']} months, mean {float(recent.mean()):.2f} C")

In [ ]:
# a look at the hourly file too — it captures the diurnal cycle
dsh = xr.open_dataset(NC_HOURLY).squeeze(drop=True)
dsh["t_c"] = dsh["t"] - 273.15

t_idx = pd.DatetimeIndex(dsh.valid_time.values)
print("The hourly file")
print(f"  time steps : {dsh.sizes['valid_time']}")
print(f"  hours      : {sorted(t_idx.hour.unique())}  (UTC)")
print(f"  Bangladesh time = UTC + 6  ->  "
      f"{sorted((t_idx.hour.unique()+6) % 24)}")
print(f"  months     : {t_idx.to_period('M').nunique()}")
print(f"\n  meaning: for each month, a mean over {len(t_idx.hour.unique())} hours ->"
      f" {t_idx.to_period('M').nunique()} × {len(t_idx.hour.unique())}"
      f" = {dsh.sizes['valid_time']} ✓")
print("\n-> This is the ERA5 'monthly averaged by hour of day' product.")
print("  It tells us how warm it is at each hour of the day (Notebook 4).")

---
# Part 5 — A first map: vector and raster together

Now we bring the two families into one figure — this is the real power of geospatial analysis.

In [ ]:
# a map of the 25-year mean temperature + the boundaries of Bangladesh
t_mean = ds.t_c.mean("valid_time")

fig, ax = plt.subplots(figsize=(7.5, 6.6))
im = t_mean.plot(ax=ax, cmap="RdYlBu_r", add_colorbar=False)
cb = plt.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
cb.set_label("Mean air temperature (°C)", fontsize=10)

# the national boundary on top
bbs[0].boundary.plot(ax=ax, color="k", linewidth=1.5)
bbs[1].boundary.plot(ax=ax, color="k", linewidth=0.5, alpha=0.6)

ax.set_title("ERA5 mean temperature at 1000 hPa, 2000–2024\n"
             "with Bangladesh administrative boundaries", fontsize=11)
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

print(f"Mean temperature of the grid: {float(t_mean.mean()):.2f} C")
print("Notice — the grid is larger than Bangladesh; it also covers India, Myanmar and the Bay of Bengal.")
print("Cutting out only the Bangladesh part is the job of Notebook 3 (clipping).")

In [ ]:
# the four seasonal means — four panels in one figure
seasons = {"Winter (DJF)": [12, 1, 2], "Pre-monsoon (MAM)": [3, 4, 5],
           "Monsoon (JJAS)": [6, 7, 8, 9], "Post-monsoon (ON)": [10, 11]}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
vmin, vmax = 16, 31
for a, (name, months) in zip(axes.ravel(), seasons.items()):
    sub = ds.t_c.sel(valid_time=ds.valid_time.dt.month.isin(months)).mean("valid_time")
    im = sub.plot(ax=a, cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False)
    bbs[0].boundary.plot(ax=a, color="k", linewidth=1.1)
    a.set_title(f"{name}   (mean {float(sub.mean()):.1f} °C)", fontsize=10)
    a.set_xlabel(""); a.set_ylabel("")

fig.colorbar(im, ax=axes, shrink=0.6, label="Temperature (°C)", pad=0.02)
fig.suptitle("Seasonal mean temperature over Bangladesh, ERA5 2000–2024",
             fontsize=12, fontweight="bold", y=0.97)
plt.show()

In [ ]:
# a 25-year time series for a single place
places = {"Dhaka": (23.81, 90.41), "Chattogram": (22.36, 91.78),
          "Rajshahi": (24.37, 88.60), "Sylhet": (24.90, 91.87)}

fig, ax = plt.subplots(figsize=(12, 4.2))
for (name, (la, lo)), col in zip(places.items(), [C_WARM, C_COOL, C_ACC, "#7FB069"]):
    s = ds.t_c.sel(latitude=la, longitude=lo, method="nearest")
    ax.plot(s.valid_time, s.values, lw=0.8, label=name, color=col, alpha=0.9)

ax.set_title("Monthly mean temperature at four cities, ERA5 2000–2024")
ax.set_xlabel("Year"); ax.set_ylabel("Temperature (°C)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.tight_layout(); plt.show()

print("A clear seasonal oscillation — low in winter, high in summer.")
print("Hidden beneath that oscillation is the long-term trend — we extract it in Notebook 4.")

In [ ]:
# saving today's work — the later notebooks will use these
dist.to_parquet(OUT/"bd_districts.parquet")
bbs[1][["ADM1_EN", "ADM1_PCODE", "geometry"]].to_parquet(OUT/"bd_divisions.parquet")
bbs[0][["ADM0_EN", "geometry"]].to_parquet(OUT/"bd_country.parquet")

t_mean.to_netcdf(OUT/"era5_tmean_2000_2024.nc")

print("Files saved (in the outputs/ folder):")
for f in sorted(OUT.glob("bd_*.parquet")) + [OUT/"era5_tmean_2000_2024.nc"]:
    print(f"  {f.name:34s} {f.stat().st_size/1e6:7.2f} MB")

---
# Home Work — Class 08

**1.** Read `bbs[3]` (the upazila level). How many upazilas are there? Find how many fall in
Dhaka division (`ADM1_EN == "Dhaka"`).

**2.** Build a `Polygon` that encloses the approximate area of the Sundarbans
(longitude 89.0–89.9, latitude 21.6–22.4). Print its `area`, `centroid` and
`bounds`. Then find which districts `intersects` it.

**3.** From `ds.r` (relative humidity), draw a map of the 25-year mean —
with the boundaries of Bangladesh on top. Use the `"YlGnBu"` colormap.

**4.** Write the district data to both **GeoJSON** and **GeoParquet** and compare the sizes.
How many times smaller is one than the other?

**5.** From `ds.t_c`, take the mean of the twelve months of **2024 only**, and draw a map of the
difference from the 2000 mean (2024 − 2000). Where has it warmed most?

---
# Class 08 — Summary

| What we learned | Key command |
|-----------|-------------|
| building an environment | `conda create -n geo -c conda-forge geopandas ...` |
| reading vector data | `gpd.read_file("x.shp")` |
| writing vector data | `.to_file()` · `.to_parquet()` |
| creating geometry | `Point()` `LineString()` `Polygon()` |
| measuring geometry | `.area` `.length` `.centroid` `.bounds` |
| geometric relationships | `.contains()` `.within()` `.intersects()` |
| geometric operations | `.buffer()` `.intersection()` `.union()` `.difference()` |
| reading NetCDF | `xr.open_dataset("x.nc")` |
| selecting from a grid | `.sel()` `.isel()` `.sel(method="nearest")` |
| drawing a map | `.plot()` + `gdf.boundary.plot()` |

---

### The most common mistakes

1. **Copying only the `.shp` file** — without `.shx`, `.dbf` and `.prj` it is useless.
2. **Measuring area in degrees** — square degrees mean nothing; you need a projected CRS (Notebook 2).
3. **Forgetting kelvin → Celsius** — every result will be 273 degrees too high.
4. **Mixing GADM and BBS** — use **one** boundary source in a study, and state it in Methods.
5. **Not dropping the `pressure_level` axis** — without `.squeeze()` the shapes cause trouble later.

### What comes next

**Class 09 — Vector Data Manipulation**: coordinate reference systems (CRS), correct
areas and distances, a spatial join to match the ERA5 grid to the districts,
and geocoding.

---

**Md Wabidur Rahman** — Researcher, Research and Integrated Thought (RIT)
*Geospatial Data Analysis and Visualization · Class 08 — Foundations of Spatial Data*